# Avaliação do Modelo de IA - Grupo 01
**Entregável da Semana - Residência em IA**

Este notebook executa e avalia o pipeline multiagente (LangGraph + LM Studio) responsável pela extração de argumentos, classificação socrática e síntese (Dossiê).

In [ ]:
import sys
import os
from pathlib import Path

# Configurar import path
sys.path.insert(0, str(Path(os.getcwd()).resolve()))

from src.state import initial_state
from src.graph import sistema_multiagente
from src.medicao import executar
import time

## 1. Carregamento do Modelo
A inicialização do modelo acontece nos módulos de serviço (`src.services.llm`), onde a configuração aponta para o modelo LLM executado localmente (ex: Qwen via LM Studio). A arquitetura foi desenhada em multiagentes, então o 'modelo' avaliado aqui é o orquestramento dos prompts e as saídas das LLMs.

In [ ]:
def avaliar_pipeline(texto_teste):
    print(f"Iniciando avaliação para o texto:\n'{texto_teste[:100]}...'\n")
    start_time = time.time()
    
    estado_inicial = initial_state(texto_teste)
    
    try:
        execucao = executar(
            sistema_multiagente,
            estado_inicial,
            ao_terminar_no=lambda no, segundos: print(f"Agente [{no}] concluído em {segundos:.2f}s")
        )
        end_time = time.time()
        print(f"\nTempo total de execução: {end_time - start_time:.2f} segundos")
        
        estado_final = execucao.estado
        return estado_final
        
    except Exception as e:
        print(f"Falha na execução: {e}")
        return None

## 2. Execução e Validação (Testes)

In [ ]:
texto_amostra_1 = """
O avião da FAB que desapareceu em SC foi encontrado totalmente destruído nesta tarde.
Segundo fontes, foi uma tragédia terrível. Dois corpos foram encontrados carbonizados.
"""

resultado = avaliar_pipeline(texto_amostra_1)

In [ ]:
if resultado:
    print("\n=== RELATÓRIO SINTETIZADO (DOSSIÊ) ===\n")
    print(resultado.dossier)

## 3. Avaliação de Métricas
A principal métrica de sucesso para este estágio é a **Taxa de Adequação do Guardrail** e **Resiliência do Pipeline** (Degradação Graciosa).

*   O agente deve suprimir falsos positivos em adjetivação extrema (Ex: "carbonizado" em acidentes).
*   O agente deve lidar com a ausência de base de dados e retornar amigavelmente.
*   O tempo de inferência total deve se manter abaixo de 90 segundos na CPU local.